# Observability and Serving Simulation

Trainer mode: we treat this like an ops incident drill.

**Outcome:** capture telemetry and explain latency behavior.

## Step 1 - Quick telemetry snapshot

Pause and ask: what utilization value would trigger your alert?

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

In [ ]:
import time
import torch

## Step 2 - Simulate inference load

Expected: repeated forward passes reveal latency trend.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.nn.Linear(1024, 1024).to(device)

In [ ]:
batch = torch.randn((64, 1024), device=device)
if device == "cuda":
    torch.cuda.synchronize()

In [ ]:
t0 = time.perf_counter()
for _ in range(30):
    _ = model(batch)

In [ ]:
if device == "cuda":
    torch.cuda.synchronize()
lat = time.perf_counter() - t0
print(round(lat, 4))

## Step 3 - Interpretation

Ask learners:
- Is latency acceptable for your SLO?
- What would you scale first: batch, model, or hardware?

Debug hint: rerun once after warm-up for stable numbers.